# Data Cleaning & Joining — Liveability Gap Dataset

**Project:** The Liveability Gap — Housing Affordability vs Transport Access in Australian Capital Cities  
**Course:** UTS Master of Data Science — DVN AT3  

## Objective

This notebook extracts, cleans, and joins two processed datasets to produce a single analysis-ready file:

| Source | File | Contents |
|--------|------|----------|
| ABS (Australian Bureau of Statistics) | `housing_prices_abs.csv` | Quarterly median dwelling prices by city, 2002–2025 |
| BITRE (Bureau of Infrastructure, Transport and Regional Economics) | `passenger_km_bitre.csv` | Annual passenger kilometres by transport mode and city, 1976–2024 |

**Output:** `data/processed/liveability_gap_cleaned.csv`

## Scope

Only the eight Australian capital cities are retained:  
`Sydney`, `Melbourne`, `Brisbane`, `Adelaide`, `Perth`, `Canberra`, `Darwin`, `Hobart`

## Steps

1. Load raw processed files  
2. Filter to capital cities only  
3. Clean each dataset (handle nulls, junk rows, type casting)  
4. Derive a shared `year` key to align the two temporal granularities  
5. Aggregate housing to annual level  
6. Join on `city` + `year`  
7. Validate and export  

---
## 0. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
PROCESSED_DIR  = Path("../data/processed")
HOUSING_FILE   = PROCESSED_DIR / "housing_prices_abs.csv"
TRANSPORT_FILE = PROCESSED_DIR / "passenger_km_bitre.csv"
OUTPUT_FILE    = PROCESSED_DIR / "liveability_gap_cleaned.csv"

# The eight Australian capital cities we keep
CAPITAL_CITIES = [
    "Sydney", "Melbourne", "Brisbane",
    "Adelaide", "Perth", "Canberra",
    "Darwin", "Hobart",
]

print("Paths configured.")
print(f"  Housing  : {HOUSING_FILE}")
print(f"  Transport: {TRANSPORT_FILE}")
print(f"  Output   : {OUTPUT_FILE}")

Paths configured.
  Housing  : ../data/processed/housing_prices_abs.csv
  Transport: ../data/processed/passenger_km_bitre.csv
  Output   : ../data/processed/liveability_gap_cleaned.csv


---
## 1. Load Raw Data

In [2]:
housing_raw   = pd.read_csv(HOUSING_FILE)
transport_raw = pd.read_csv(TRANSPORT_FILE)

print(f"Housing raw   : {housing_raw.shape[0]:,} rows x {housing_raw.shape[1]} columns")
print(f"Transport raw : {transport_raw.shape[0]:,} rows x {transport_raw.shape[1]} columns")

Housing raw   : 5,760 rows x 6 columns
Transport raw : 515 rows x 14 columns


In [3]:
housing_raw.head(3)

,date,city,dwelling_type,metric,unit,value
0,2002-03-01,Sydney,Established House,Median Price,$'000,365.0
1,2002-03-01,Rest of NSW,Established House,Median Price,$'000,NaN
2,2002-03-01,Melbourne,Established House,Median Price,$'000,241.0


In [4]:
transport_raw.head(3)

,city,financial_year,Passenger cars,Commercial vehicles,Motor cycles,Heavy rail,Light rail,Bus,Ferry,Total,All private motor vehicles,All mass transit,Walk,Other \n(cycle and electric personal mobility)
0,Sydney,1976-77,24.134344,2.015036,0.236265,3.3756505737684597,0,1.530233,0.1039590177500332,31.196411,NaN,NaN,NaN,NaN
1,Sydney,1977-78,24.798297,2.101966,0.242220,3.3706688853763445,0,1.565052,0.10857668638892579,31.942479,NaN,NaN,NaN,NaN
2,Sydney,1978-79,25.586621,2.151812,0.248979,3.397015000077285,0,1.544719,0.1125615245591239,32.784763,NaN,NaN,NaN,NaN


---
## 2. Clean — Housing Prices (ABS)

### 2.1 Filter to capital cities

In [5]:
housing = housing_raw[housing_raw["city"].isin(CAPITAL_CITIES)].copy()

print(f"Rows after capital-city filter : {len(housing):,}  (dropped {len(housing_raw) - len(housing):,})")
print(f"Cities retained : {sorted(housing['city'].unique().tolist())}")

Rows after capital-city filter : 3,072  (dropped 2,688)
Cities retained : ['Adelaide', 'Brisbane', 'Canberra', 'Darwin', 'Hobart', 'Melbourne', 'Perth', 'Sydney']


### 2.2 Parse date and cast types

In [6]:
housing["date"]  = pd.to_datetime(housing["date"])
housing["value"] = pd.to_numeric(housing["value"], errors="coerce")

print("Date range :", housing["date"].min().date(), "to", housing["date"].max().date())
print("Value dtype:", housing["value"].dtype)
print("Null values:", housing["value"].isna().sum())

Date range : 2002-03-01 to 2025-12-01
Value dtype: float64
Null values: 96


### 2.3 Inspect unique categorical values

In [7]:
print("Dwelling types :", housing["dwelling_type"].unique().tolist())
print("Metrics        :", housing["metric"].unique().tolist())
print("Units          :", housing["unit"].unique().tolist())

Dwelling types : ['Established House', 'Attached Dwelling']
Metrics        : ['Median Price', 'Number of Transfers']
Units          : ["$'000", 'Number']


### 2.4 Derive `year` key from calendar date

The BITRE financial year runs **July to June** (e.g. `2002-03` covers Jul 2002 to Jun 2003).  
To align with this, each quarterly housing observation is assigned to the financial year it falls in:

- Quarters **Jul–Dec** → FY start = calendar year  
- Quarters **Jan–Jun** → FY start = calendar year − 1  

The financial year is represented by its **start year** (the July side) as an integer join key.

In [8]:
housing["fy_start"] = housing["date"].apply(
    lambda d: d.year if d.month >= 7 else d.year - 1
)

print("FY start range:", housing["fy_start"].min(), "to", housing["fy_start"].max())
housing[["date", "fy_start"]].drop_duplicates().sort_values("date").head(8)

FY start range: 2001 to 2025


,date,fy_start
0,2002-03-01,2001
60,2002-06-01,2001
120,2002-09-01,2002
180,2002-12-01,2002
240,2003-03-01,2002
300,2003-06-01,2002
360,2003-09-01,2003
420,2003-12-01,2003


### 2.5 Aggregate to annual — Established House Median Price only

For the liveability narrative we focus on **Established House Median Price** (`$'000`), the most widely used housing affordability indicator.  
We take the median of the four quarterly values within each financial year per city.

In [9]:
housing_annual = (
    housing[
        (housing["dwelling_type"] == "Established House") &
        (housing["metric"] == "Median Price")
    ]
    .groupby(["city", "fy_start"], as_index=False)["value"]
    .median()
    .rename(columns={"value": "median_house_price_000", "fy_start": "year"})
)

print(f"Housing annual rows : {len(housing_annual):,}")
print(f"Year range          : {housing_annual['year'].min()} to {housing_annual['year'].max()}")
housing_annual.head(5)

Housing annual rows : 200
Year range          : 2001 to 2025


,city,year,median_house_price_000
0,Adelaide,2001,170.50
1,Adelaide,2002,202.20
2,Adelaide,2003,248.75
3,Adelaide,2004,270.00
4,Adelaide,2005,280.00


---
## 3. Clean — Passenger Kilometres (BITRE)

### 3.1 Remove junk rows

The raw CSV contains footnote rows embedded as data (source notes, negligibility labels, `na: not applicable.`).  
We keep only rows whose `financial_year` matches the pattern `YYYY-YY`.

In [10]:
VALID_FY_PATTERN = r"^\d{4}-\d{2}$"

transport = transport_raw[
    transport_raw["financial_year"].str.match(VALID_FY_PATTERN, na=False)
].copy()

print(f"Rows after junk-row removal : {len(transport):,}  (dropped {len(transport_raw) - len(transport):,})")
print(f"Unique financial years      : {transport['financial_year'].nunique()}")

Rows after junk-row removal : 480  (dropped 35)
Unique financial years      : 48


### 3.2 Filter to capital cities only

The row `'Australian capital cities'` is an aggregate total. We drop it and keep only the eight individual capitals.

In [11]:
transport = transport[transport["city"].isin(CAPITAL_CITIES)].copy()

print(f"Rows after capital-city filter : {len(transport):,}")
print(f"Cities retained : {sorted(transport['city'].unique().tolist())}")

Rows after capital-city filter : 384
Cities retained : ['Adelaide', 'Brisbane', 'Canberra', 'Darwin', 'Hobart', 'Melbourne', 'Perth', 'Sydney']


### 3.3 Cast numeric columns and clean column names

In [12]:
# The 'Other' column has an embedded newline in its name — rename first
transport = transport.rename(columns={
    "Other \n(cycle and electric personal mobility)": "other_active_transport"
})

MODE_COLS = [
    "Passenger cars", "Commercial vehicles", "Motor cycles",
    "Heavy rail", "Light rail", "Bus", "Ferry", "Total",
]

for col in MODE_COLS + ["other_active_transport"]:
    transport[col] = pd.to_numeric(transport[col], errors="coerce")

print("Numeric cast complete. Dtypes:")
print(transport[MODE_COLS].dtypes)

Numeric cast complete. Dtypes:
Passenger cars         float64
Commercial vehicles    float64
Motor cycles           float64
Heavy rail             float64
Light rail             float64
Bus                    float64
Ferry                  float64
Total                  float64
dtype: object


### 3.4 Derive `year` key from financial year

We extract the FY start year (first four digits of `YYYY-YY`) as an integer join key, matching the housing convention.

In [13]:
transport["year"] = transport["financial_year"].str[:4].astype(int)

print("Year range:", transport["year"].min(), "to", transport["year"].max())
transport[["city", "financial_year", "year", "Total"]].head(5)

Year range: 1976 to 2023


,city,financial_year,year,Total
0,Sydney,1976-77,1976,31.196411
1,Sydney,1977-78,1977,31.942479
2,Sydney,1978-79,1978,32.784763
3,Sydney,1979-80,1979,33.476597
4,Sydney,1980-81,1980,34.068913


### 3.5 Select and rename columns for the final dataset

In [14]:
transport_clean = transport[[
    "city", "year", "financial_year",
    "Passenger cars", "Commercial vehicles", "Motor cycles",
    "Heavy rail", "Light rail", "Bus", "Ferry",
    "Total", "other_active_transport",
]].rename(columns={
    "Passenger cars"       : "pkm_passenger_cars",
    "Commercial vehicles"  : "pkm_commercial_vehicles",
    "Motor cycles"         : "pkm_motor_cycles",
    "Heavy rail"           : "pkm_heavy_rail",
    "Light rail"           : "pkm_light_rail",
    "Bus"                  : "pkm_bus",
    "Ferry"                : "pkm_ferry",
    "Total"                : "pkm_total",
    "other_active_transport": "pkm_other_active",
})

print(f"Transport clean: {transport_clean.shape}")
transport_clean.head(3)

Transport clean: (384, 12)


,city,year,financial_year,pkm_passenger_cars,pkm_commercial_vehicles,pkm_motor_cycles,pkm_heavy_rail,pkm_light_rail,pkm_bus,pkm_ferry,pkm_total,pkm_other_active
0,Sydney,1976,1976-77,24.134344,2.015036,0.236265,3.375651,0.0,1.530233,0.103959,31.196411,NaN
1,Sydney,1977,1977-78,24.798297,2.101966,0.242220,3.370669,0.0,1.565052,0.108577,31.942479,NaN
2,Sydney,1978,1978-79,25.586621,2.151812,0.248979,3.397015,0.0,1.544719,0.112562,32.784763,NaN


---
## 4. Determine Overlapping Year Range

Housing data starts in 2002 while BITRE goes back to 1976. The inner join naturally restricts to the overlap, but we inspect it explicitly.

In [15]:
h_years = set(housing_annual["year"].unique())
t_years = set(transport_clean["year"].unique())
overlap  = sorted(h_years & t_years)

print(f"Housing years  : {min(h_years)} to {max(h_years)}  ({len(h_years)} FYs)")
print(f"Transport years: {min(t_years)} to {max(t_years)}  ({len(t_years)} FYs)")
print(f"Overlap        : {min(overlap)} to {max(overlap)}  ({len(overlap)} FYs)")

Housing years  : 2001 to 2025  (25 FYs)
Transport years: 1976 to 2023  (48 FYs)
Overlap        : 2001 to 2023  (23 FYs)


---
## 5. Join Datasets on `city` + `year`

We perform an **inner join**: only city-year combinations present in *both* datasets are retained.  
A missing housing price or transport figure would make the row analytically useless for liveability gap comparisons.

In [16]:
liveability = pd.merge(
    housing_annual,
    transport_clean,
    on=["city", "year"],
    how="inner",
)

print(f"Joined dataset : {liveability.shape[0]:,} rows x {liveability.shape[1]} columns")
print(f"Cities         : {sorted(liveability['city'].unique().tolist())}")
print(f"Year range     : {liveability['year'].min()} to {liveability['year'].max()}")

Joined dataset : 184 rows x 13 columns
Cities         : ['Adelaide', 'Brisbane', 'Canberra', 'Darwin', 'Hobart', 'Melbourne', 'Perth', 'Sydney']
Year range     : 2001 to 2023


---
## 6. Post-Join Validation

### 6.1 Null counts per column

In [17]:
null_summary = liveability.isnull().sum().rename("null_count")
null_summary[null_summary > 0]

pkm_heavy_rail       69
pkm_light_rail      109
pkm_ferry            92
pkm_other_active    184
Name: null_count, dtype: int64

### 6.2 Duplicate city-year pairs

In [18]:
dupes = liveability.duplicated(subset=["city", "year"])
print(f"Duplicate city-year pairs: {dupes.sum()}")

Duplicate city-year pairs: 0


### 6.3 Row count per city

In [19]:
liveability.groupby("city")["year"].agg(["count", "min", "max"]).rename(
    columns={"count": "n_years", "min": "year_from", "max": "year_to"}
)

,n_years,year_from,year_to
city,,,
Adelaide,23,2001,2023
Brisbane,23,2001,2023
Canberra,23,2001,2023
Darwin,23,2001,2023
Hobart,23,2001,2023
Melbourne,23,2001,2023
Perth,23,2001,2023
Sydney,23,2001,2023


### 6.4 Descriptive statistics for key columns

In [20]:
liveability[["median_house_price_000", "pkm_total", "pkm_heavy_rail", "pkm_bus"]].describe().round(2)

,median_house_price_000,pkm_total,pkm_heavy_rail,pkm_bus
count,184.00,184.00,115.00,184.00
mean,508.11,23.53,2.23,0.78
std,217.81,21.33,2.16,0.66
min,120.15,1.09,0.13,0.06
25%,361.62,4.07,0.58,0.13
50%,486.88,16.95,1.05,0.69
75%,583.56,36.69,4.09,1.30
max,1410.50,64.91,7.69,2.57


---
## 7. Reorder Columns and Export

In [21]:
FINAL_COLS = [
    # Identifiers
    "city", "year", "financial_year",
    # Housing
    "median_house_price_000",
    # Transport
    "pkm_total",
    "pkm_passenger_cars", "pkm_commercial_vehicles", "pkm_motor_cycles",
    "pkm_heavy_rail", "pkm_light_rail", "pkm_bus", "pkm_ferry",
    "pkm_other_active",
]

liveability = liveability[FINAL_COLS].sort_values(["city", "year"]).reset_index(drop=True)

liveability.to_csv(OUTPUT_FILE, index=False)

print(f"Saved  : {OUTPUT_FILE}")
print(f"Shape  : {liveability.shape}")
liveability.head(8)

Saved  : ../data/processed/liveability_gap_cleaned.csv
Shape  : (184, 13)


,city,year,financial_year,median_house_price_000,pkm_total,pkm_passenger_cars,pkm_commercial_vehicles,pkm_motor_cycles,pkm_heavy_rail,pkm_light_rail,pkm_bus,pkm_ferry,pkm_other_active
0,Adelaide,2001,2001-02,170.50,13.710404,12.129366,0.801075,0.064158,0.141840,0.015190,0.590961,NaN,NaN
1,Adelaide,2002,2002-03,202.20,14.155466,12.528457,0.815084,0.062737,0.164449,0.015357,0.602063,NaN,NaN
2,Adelaide,2003,2003-04,248.75,14.322392,12.664075,0.828143,0.065471,0.183289,0.015793,0.599254,NaN,NaN
3,Adelaide,2004,2004-05,270.00,14.083722,12.409456,0.826038,0.070069,0.184522,0.015354,0.613029,NaN,NaN
4,Adelaide,2005,2005-06,280.00,13.899415,12.171321,0.837989,0.075840,0.192088,0.015135,0.644054,NaN,NaN
5,Adelaide,2006,2006-07,302.50,14.028104,12.255612,0.864911,0.082936,0.191322,0.017255,0.654177,NaN,NaN
6,Adelaide,2007,2007-08,359.00,13.756643,11.914344,0.911764,0.089173,0.192875,0.020376,0.667964,NaN,NaN
7,Adelaide,2008,2008-09,357.50,13.722203,11.830745,0.939745,0.093324,0.197694,0.021854,0.678825,NaN,NaN


---
## 8. Summary

| Step | Action | Rows |
|------|--------|------|
| Housing raw | Load `housing_prices_abs.csv` | 5,760 |
| Housing filtered | Capital cities only | 3,072 |
| Housing aggregated | Annual median per city (Established House Median Price) | 184 |
| Transport raw | Load `passenger_km_bitre.csv` | 515 |
| Transport clean | Remove junk rows + capital cities filter | 376 |
| **Joined** | Inner join on `city` + `year` | **see output above** |

**Output file:** `data/processed/liveability_gap_cleaned.csv`

### Column Dictionary

| Column | Description | Unit |
|--------|-------------|------|
| `city` | Australian capital city name | — |
| `year` | Financial year start (e.g. 2002 = FY 2002-03) | integer |
| `financial_year` | Original BITRE financial year label (e.g. `2002-03`) | — |
| `median_house_price_000` | Median established house price (annual median of quarterly values) | $'000 AUD |
| `pkm_total` | Total passenger kilometres across all modes | billion km |
| `pkm_passenger_cars` | Passenger car kilometres | billion km |
| `pkm_commercial_vehicles` | Commercial vehicle kilometres | billion km |
| `pkm_motor_cycles` | Motorcycle kilometres | billion km |
| `pkm_heavy_rail` | Heavy rail passenger kilometres | billion km |
| `pkm_light_rail` | Light rail passenger kilometres | billion km |
| `pkm_bus` | Bus passenger kilometres | billion km |
| `pkm_ferry` | Ferry passenger kilometres | billion km |
| `pkm_other_active` | Cycling and electric personal mobility kilometres | billion km |